# Biological Databases

Modern bioinformatics depends on large, well-curated public databases. This notebook covers the major databases you will use daily as a bioinformatician: where biological data lives, what each database contains, and how to access it programmatically.

## Learning Objectives

- Navigate the NCBI suite of databases (GenBank, RefSeq, Gene, SRA, GEO)
- Understand UniProt (SwissProt vs TrEMBL) and search for protein information
- Use the PDB to find and evaluate protein structures
- Access Ensembl genome browser and BioMart
- Retrieve data programmatically using Bio.Entrez and REST APIs

**Prerequisites:** Basic Python, familiarity with DNA/RNA/protein sequences

<!-- COPILOT_NOTEBOOK_ONBOARDING_V1 -->
## Why this notebook matters in real projects
The methods here are applied in sequence analysis, annotation, motif/protein interpretation, and evidence building from biological databases.


## How to start using this notebook
1. Run the **Quick starter data demo** cell below first to confirm your environment and file paths.
2. Then run notebook sections top-to-bottom once, and only then start modifying parameters.
3. Keep one small, known-good example input while experimenting; compare each output against it.
4. For larger or real data, copy the same workflow and replace only input paths first.


## Complicated moments explained
These are common points where learners usually get stuck:
- Similarity is not identity: alignments, scores, and biological function are related but not equivalent.
- Database provenance matters: always track which release/version generated your result.
- Threshold choices (e-value, identity, score) strongly change sensitivity vs specificity.


## Quick starter data demo (run this first)
This cell loads tiny synthetic files from `Course/Assets/data/notebook_starters/` so you can see real input/output behavior immediately.


In [ ]:
# Quick starter demo: load tiny example files and inspect outputs
from pathlib import Path
import json
import csv

root = Path.cwd().resolve()
while root != root.parent and not (root / 'Course').exists():
    root = root.parent

base = root / 'Course' / 'Assets' / 'data' / 'notebook_starters'
print(f'Using starter data folder: {base}')

with open(base / 'dna_sequences.csv', newline='') as f:
    rows = list(csv.DictReader(f))
print('dna_sequences.csv (first 3 rows):')
for row in rows[:3]:
    print(row)

with open(base / 'variants.csv', newline='') as f:
    variants = list(csv.DictReader(f))
print('\nvariants.csv (first 3 rows):')
for row in variants[:3]:
    print(row)

print('\nprotein_sequences.fasta (headers):')
for line in (base / 'protein_sequences.fasta').read_text().splitlines():
    if line.startswith('>'):
        print(line)

meta = json.loads((base / 'sample_metadata.json').read_text())
print('\nMetadata keys:', list(meta.keys()))


---

[← Previous: Bioinformatics Glossary: A-Z Reference](../../Tier_2_Core_Bioinformatics/00_Skills_Check/01_glossary.ipynb) | [Next: BioPython Essentials →](../../Tier_2_Core_Bioinformatics/02_BioPython_Essentials/01_biopython_essentials.ipynb)

---

In [ ]:
# Install BioPython if needed
# !pip install biopython

from Bio import Entrez, SeqIO
import urllib.request
import json

# IMPORTANT: Always set your email when using NCBI Entrez
Entrez.email = "your.email@example.com"  # Replace with your real email

print("Ready to explore biological databases!")

---
## 1. The NCBI Ecosystem

The National Center for Biotechnology Information (NCBI) hosts an interconnected suite of databases. Understanding what each one contains is essential.

```
                         NCBI Databases
    ┌─────────────────────────────────────────────────────┐
    │                                                     │
    │  Sequences          Expression       Literature     │
    │  ──────────         ──────────       ──────────     │
    │  GenBank            GEO              PubMed         │
    │  RefSeq             SRA              PMC            │
    │  Gene                                               │
    │                                                     │
    │  Taxonomy           Variation        Structure      │
    │  ──────────         ──────────       ──────────     │
    │  Taxonomy           dbSNP            MMDB           │
    │  BioProject         ClinVar                         │
    │  BioSample          dbVar                           │
    │                                                     │
    └─────────────────────────────────────────────────────┘
         All linked via Entrez -- one search interface
```

### 1.1 GenBank

GenBank is the primary **nucleotide sequence** database. It is part of the International Nucleotide Sequence Database Collaboration (INSDC) along with ENA (Europe) and DDBJ (Japan). All three synchronize daily.

**Key facts:**
- Contains sequences submitted by researchers worldwide
- Submissions are **not curated** -- the submitter is responsible for accuracy
- Each entry has an accession number (e.g., `AB000263.1`)
- Records include sequence, annotations, features (CDS, gene, mRNA), and references

```
GenBank flat file structure:

  LOCUS       AB000263     368 bp    mRNA    linear   PRI
  DEFINITION  Homo sapiens mRNA for prepro cortistatin...
  ACCESSION   AB000263
  VERSION     AB000263.1
  FEATURES
       source  1..368
               /organism="Homo sapiens"
       CDS     43..180
               /product="prepro-cortistatin"
  ORIGIN
        1 atggcctctg cagcagacga cc...
  //
```

### 1.2 RefSeq

RefSeq is NCBI's **curated** sequence database. Unlike GenBank, RefSeq entries are reviewed and non-redundant.

| Prefix | Molecule Type | Curation Level |
|--------|--------------|----------------|
| NM_ | mRNA | Curated |
| NR_ | non-coding RNA | Curated |
| NP_ | protein | Curated |
| NC_ | complete genomic molecule | Curated |
| NG_ | genomic region | Curated |
| XM_ | mRNA | Predicted (model) |
| XP_ | protein | Predicted (model) |
| XR_ | non-coding RNA | Predicted (model) |
| WP_ | protein | Non-redundant |

**When to use GenBank vs RefSeq:**
- GenBank: when you need all submitted sequences (e.g., population studies)
- RefSeq: when you need one reliable reference sequence per gene

### 1.3 NCBI Gene

The Gene database provides a **gene-centric** view that links to all associated data:
- RefSeq transcripts and proteins
- Genomic context and neighboring genes
- Orthologs and homologs
- Expression data, pathways, and phenotypes
- Literature references

Each gene has a unique Gene ID (e.g., Gene ID 672 = BRCA1).

URL pattern: `https://www.ncbi.nlm.nih.gov/gene/<GeneID>`

### 1.4 SRA (Sequence Read Archive)

SRA stores **raw sequencing reads** from next-generation sequencing experiments.

- Largest publicly available repository of sequencing data
- Contains reads from Illumina, PacBio, Oxford Nanopore, etc.
- Organized by: Study (SRP) > Sample (SRS) > Experiment (SRX) > Run (SRR)
- Download with `fastq-dump` or `fasterq-dump` from the SRA Toolkit

**Example:** SRR1234567 is a specific sequencing run you can download.

### 1.5 GEO (Gene Expression Omnibus)

GEO stores **processed** gene expression data:
- Microarray data
- RNA-seq quantifications
- ChIP-seq peak calls
- Methylation arrays

GEO accession types:
- **GSE** (Series): a complete experiment with multiple samples
- **GSM** (Sample): data from one biological sample
- **GPL** (Platform): the array/technology description
- **GDS** (DataSet): curated, analysis-ready datasets

---
## 2. Programmatic Access with Bio.Entrez

The Entrez Programming Utilities (E-utilities) let you search and retrieve data from all NCBI databases via a single API. BioPython wraps these in `Bio.Entrez`.

The three main functions:

| Function | Purpose |
|----------|---------|
| `Entrez.einfo()` | Get database information |
| `Entrez.esearch()` | Search a database |
| `Entrez.efetch()` | Download records |

In [ ]:
# List all available NCBI databases
handle = Entrez.einfo()
record = Entrez.read(handle)
handle.close()

db_list = record["DbList"]
print(f"NCBI has {len(db_list)} databases:\n")

# Display in columns
for i in range(0, len(db_list), 5):
    row = db_list[i:i+5]
    print("  ".join(f"{db:<18}" for db in row))

In [ ]:
# Get details about a specific database
handle = Entrez.einfo(db="nucleotide")
record = Entrez.read(handle)
handle.close()

info = record["DbInfo"]
print(f"Database: {info['DbName']}")
print(f"Description: {info['Description']}")
print(f"Total records: {info['Count']}")
print(f"Last updated: {info['LastUpdate']}")
print(f"\nSearchable fields ({len(info['FieldList'])}):\n")
for field in info['FieldList'][:10]:
    print(f"  [{field['Name']}] - {field['FullName']}: {field['Description']}")

### 2.1 Searching NCBI with esearch()

The `esearch()` function searches a database and returns matching record IDs.

**Search syntax tips:**
- `insulin[Gene]` -- search in the Gene field
- `Homo sapiens[Organism]` -- search by organism
- `AND`, `OR`, `NOT` -- Boolean operators
- `RefSeq[Filter]` -- limit to RefSeq entries
- `2020:2024[PDAT]` -- publication date range

In [ ]:
# Search for human insulin mRNA sequences in RefSeq
handle = Entrez.esearch(
    db="nucleotide",
    term="insulin[Gene] AND Homo sapiens[Organism] AND mRNA[Filter] AND RefSeq[Filter]",
    retmax=10
)
results = Entrez.read(handle)
handle.close()

print(f"Total matches: {results['Count']}")
print(f"IDs returned: {results['IdList']}")
print(f"\nNote: retmax={10}, so up to 10 IDs are returned.")
print(f"Set retmax higher to get more IDs.")

In [ ]:
# Search PubMed for recent bioinformatics papers
handle = Entrez.esearch(
    db="pubmed",
    term="CRISPR AND bioinformatics AND 2023:2024[PDAT]",
    retmax=5
)
results = Entrez.read(handle)
handle.close()

print(f"Total CRISPR + bioinformatics papers (2023-2024): {results['Count']}")
print(f"First 5 PubMed IDs: {results['IdList']}")

### 2.2 Fetching Records with efetch()

Once you have IDs from `esearch()`, use `efetch()` to download actual records.

Key parameters:
- `db` -- which database
- `id` -- accession or UID (can be a comma-separated list)
- `rettype` -- return type: `"fasta"`, `"gb"` (GenBank), `"gp"` (GenPept)
- `retmode` -- return mode: `"text"` or `"xml"`

In [ ]:
# Fetch a sequence in FASTA format
handle = Entrez.efetch(
    db="nucleotide",
    id="NM_000207.3",   # Human insulin mRNA
    rettype="fasta",
    retmode="text"
)
record = SeqIO.read(handle, "fasta")
handle.close()

print(f"ID: {record.id}")
print(f"Description: {record.description}")
print(f"Length: {len(record)} bp")
print(f"First 60 nt: {record.seq[:60]}")

In [ ]:
# Fetch in GenBank format to get full annotations
handle = Entrez.efetch(
    db="nucleotide",
    id="NM_000207.3",
    rettype="gb",
    retmode="text"
)
record = SeqIO.read(handle, "genbank")
handle.close()

print(f"ID: {record.id}")
print(f"Name: {record.name}")
print(f"Description: {record.description}")
print(f"Sequence length: {len(record)} bp")
print(f"\nAnnotations:")
for key in ['organism', 'taxonomy', 'keywords']:
    if key in record.annotations:
        print(f"  {key}: {record.annotations[key]}")

print(f"\nFeatures ({len(record.features)}):")
for feat in record.features:
    print(f"  {feat.type:12s} {feat.location}")

In [ ]:
# Extract CDS from the GenBank record and translate it
for feature in record.features:
    if feature.type == "CDS":
        # Get the CDS nucleotide sequence
        cds_seq = feature.location.extract(record.seq)
        print(f"CDS location: {feature.location}")
        print(f"CDS length: {len(cds_seq)} bp")
        
        # Get the annotated translation (from the GenBank file)
        if "translation" in feature.qualifiers:
            annotated_protein = feature.qualifiers["translation"][0]
            print(f"\nAnnotated protein ({len(annotated_protein)} aa):")
            print(f"  {annotated_protein[:60]}...")
        
        # Translate it ourselves for verification
        our_protein = cds_seq.translate(to_stop=True)
        print(f"\nOur translation ({len(our_protein)} aa):")
        print(f"  {str(our_protein)[:60]}...")
        
        # Gene name
        if "gene" in feature.qualifiers:
            print(f"\nGene: {feature.qualifiers['gene'][0]}")
        break

In [ ]:
# Fetch a PubMed abstract
handle = Entrez.efetch(
    db="pubmed",
    id="27886196",
    rettype="abstract",
    retmode="text"
)
abstract_text = handle.read()
handle.close()

print("PubMed record:")
print(abstract_text[:1000])

### 2.3 Linking Between Databases with elink()

NCBI databases are cross-linked. Use `Entrez.elink()` to find related records across databases.

In [ ]:
# Find the protein record linked to our nucleotide record
handle = Entrez.elink(
    dbfrom="nucleotide",
    db="protein",
    id="NM_000207.3"
)
link_results = Entrez.read(handle)
handle.close()

# Extract linked protein IDs
for linkset in link_results:
    for link_db in linkset["LinkSetDb"]:
        print(f"Link name: {link_db['LinkName']}")
        for link in link_db["Link"][:5]:
            print(f"  Protein ID: {link['Id']}")

---
## 3. UniProt: The Protein Knowledgebase

UniProt is the most comprehensive protein database, organized into two sections:

| Section | Curation | Size | Use when you need |
|---------|----------|------|-------------------|
| **SwissProt** | Manually reviewed | ~570,000 entries | Reliable, well-annotated protein info |
| **TrEMBL** | Automatically annotated | ~250,000,000 entries | Broader coverage, less curated |

**What UniProt provides per protein entry:**
- Function description
- Subcellular location
- Post-translational modifications
- Protein domains and families
- Disease associations
- 3D structure cross-references
- Gene Ontology (GO) annotations
- Sequence variants and isoforms

In [ ]:
# Access UniProt via their REST API
def fetch_uniprot(accession, fmt="json"):
    """Fetch a UniProt entry by accession."""
    url = f"https://rest.uniprot.org/uniprotkb/{accession}.{fmt}"
    req = urllib.request.Request(url)
    with urllib.request.urlopen(req) as response:
        if fmt == "json":
            return json.loads(response.read().decode())
        return response.read().decode()

# Fetch human insulin (SwissProt entry)
insulin = fetch_uniprot("P01308")

print(f"Protein: {insulin['proteinDescription']['recommendedName']['fullName']['value']}")
print(f"Gene: {insulin['genes'][0]['geneName']['value']}")
print(f"Organism: {insulin['organism']['scientificName']}")
print(f"Length: {insulin['sequence']['length']} aa")
print(f"\nSequence:")
seq = insulin['sequence']['value']
for i in range(0, len(seq), 60):
    print(f"  {seq[i:i+60]}")

In [ ]:
# Explore protein function and annotations
print("Function:")
for comment in insulin.get('comments', []):
    if comment['commentType'] == 'FUNCTION':
        for text in comment['texts']:
            print(f"  {text['value'][:200]}")

print("\nSubcellular location:")
for comment in insulin.get('comments', []):
    if comment['commentType'] == 'SUBCELLULAR LOCATION':
        for subloc in comment.get('subcellularLocations', []):
            loc = subloc.get('location', {}).get('value', 'N/A')
            print(f"  {loc}")

print("\nCross-references to PDB structures:")
pdb_refs = [ref for ref in insulin.get('uniProtKBCrossReferences', [])
            if ref['database'] == 'PDB']
for ref in pdb_refs[:5]:
    props = {p['key']: p['value'] for p in ref.get('properties', [])}
    print(f"  {ref['id']} - Method: {props.get('Method', 'N/A')}, "
          f"Resolution: {props.get('Resolution', 'N/A')}")

In [ ]:
# Search UniProt programmatically
def search_uniprot(query, limit=5):
    """Search UniProt and return results."""
    import urllib.parse
    encoded_query = urllib.parse.quote(query)
    url = (f"https://rest.uniprot.org/uniprotkb/search?"
           f"query={encoded_query}&size={limit}&format=json")
    req = urllib.request.Request(url)
    with urllib.request.urlopen(req) as response:
        return json.loads(response.read().decode())

# Search for reviewed human hemoglobin proteins
results = search_uniprot("hemoglobin AND organism_id:9606 AND reviewed:true", limit=5)

print("UniProt search: human hemoglobin (reviewed)\n")
for entry in results['results']:
    acc = entry['primaryAccession']
    name = entry['proteinDescription']['recommendedName']['fullName']['value']
    gene = entry['genes'][0]['geneName']['value'] if entry.get('genes') else 'N/A'
    length = entry['sequence']['length']
    print(f"  {acc} | {gene:8s} | {name} ({length} aa)")

In [ ]:
# Download a sequence from UniProt in FASTA format
fasta_text = fetch_uniprot("P01308", fmt="fasta")
print("FASTA format:")
print(fasta_text)

---
## 4. PDB: Protein Data Bank

The PDB stores experimentally determined **3D structures** of biological macromolecules.

### Experimental Methods

| Method | What it does | Resolution | Strengths |
|--------|-------------|------------|----------|
| **X-ray crystallography** | Diffract X-rays through a crystal | 1.0-3.0 A typical | High resolution, well-established |
| **Cryo-EM** | Image frozen samples with electrons | 2.0-5.0 A typical | No crystallization needed, large complexes |
| **NMR** | Nuclear magnetic resonance in solution | N/A (ensemble) | Solution state, dynamics, small proteins |
| **AlphaFold** | AI prediction | N/A (confidence) | Any protein, no experiment needed |

### Understanding Resolution

Resolution (in Angstroms) indicates the level of detail visible:
- **< 1.5 A:** Individual atoms resolved, hydrogen atoms sometimes visible
- **1.5 - 2.5 A:** Good detail, side chains well defined
- **2.5 - 3.5 A:** Backbone clear, some side chains ambiguous
- **> 3.5 A:** Only overall fold visible, use with caution

**Lower number = better resolution.**

In [ ]:
# Search PDB using their REST API
def search_pdb(query_text, max_results=5):
    """Search PDB using the RCSB search API."""
    url = "https://search.rcsb.org/rcsbsearch/v2/query"
    query = {
        "query": {
            "type": "terminal",
            "service": "full_text",
            "parameters": {
                "value": query_text
            }
        },
        "return_type": "entry",
        "request_options": {
            "paginate": {
                "start": 0,
                "rows": max_results
            }
        }
    }
    data = json.dumps(query).encode()
    req = urllib.request.Request(url, data=data, headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req) as response:
        return json.loads(response.read().decode())

# Search for insulin structures
results = search_pdb("human insulin", max_results=5)
print(f"Total insulin structures: {results.get('total_count', 'N/A')}")
print("\nFirst 5 PDB IDs:")
for hit in results.get('result_set', []):
    print(f"  {hit['identifier']}")

In [ ]:
# Get detailed information about a PDB entry
def get_pdb_info(pdb_id):
    """Get metadata for a PDB entry from the RCSB Data API."""
    url = f"https://data.rcsb.org/rest/v1/core/entry/{pdb_id}"
    req = urllib.request.Request(url)
    with urllib.request.urlopen(req) as response:
        return json.loads(response.read().decode())

# Look up a classic insulin structure
info = get_pdb_info("4INS")

print(f"PDB ID: {info['rcsb_id']}")
print(f"Title: {info['struct']['title']}")

exptl = info.get('exptl', [{}])[0]
print(f"Method: {exptl.get('method', 'N/A')}")

refine = info.get('refine', [{}])[0]
print(f"Resolution: {refine.get('ls_d_res_high', 'N/A')} A")

deposit_date = info.get('rcsb_accession_info', {}).get('deposit_date', 'N/A')
print(f"Deposit date: {deposit_date}")

In [ ]:
# Download a PDB structure file
def download_pdb(pdb_id, fmt="pdb"):
    """Download a structure file from RCSB PDB.
    
    fmt: 'pdb' for PDB format, 'cif' for mmCIF format
    """
    if fmt == "pdb":
        url = f"https://files.rcsb.org/download/{pdb_id}.pdb"
    else:
        url = f"https://files.rcsb.org/download/{pdb_id}.cif"
    
    filename = f"{pdb_id}.{fmt}"
    urllib.request.urlretrieve(url, filename)
    return filename

# Download insulin structure
filename = download_pdb("4INS")
print(f"Downloaded: {filename}")

# Show first 25 lines
with open(filename) as f:
    for i, line in enumerate(f):
        if i >= 25:
            break
        print(line.rstrip())

In [ ]:
# Parse PDB file to extract basic information
def parse_pdb_atoms(filename):
    """Extract ATOM records from a PDB file."""
    atoms = []
    for line in open(filename):
        if line.startswith("ATOM"):
            atoms.append({
                "serial": int(line[6:11]),
                "name": line[12:16].strip(),
                "resname": line[17:20].strip(),
                "chain": line[21],
                "resseq": int(line[22:26]),
                "x": float(line[30:38]),
                "y": float(line[38:46]),
                "z": float(line[46:54]),
                "bfactor": float(line[60:66]),
            })
    return atoms

atoms = parse_pdb_atoms("4INS.pdb")
print(f"Total ATOM records: {len(atoms)}")

# Unique chains
chains = set(a["chain"] for a in atoms)
print(f"Chains: {sorted(chains)}")

# Residues per chain
for chain in sorted(chains):
    residues = set(a["resseq"] for a in atoms if a["chain"] == chain)
    print(f"  Chain {chain}: {len(residues)} residues")

# Show first 5 CA atoms
print("\nFirst 5 CA atoms:")
ca_atoms = [a for a in atoms if a["name"] == "CA"]
for a in ca_atoms[:5]:
    print(f"  Chain {a['chain']} {a['resname']}{a['resseq']}: "
          f"({a['x']:.2f}, {a['y']:.2f}, {a['z']:.2f}) B={a['bfactor']:.1f}")

---
## 5. Ensembl

Ensembl (maintained by EBI/EMBL) provides **genome annotation** for vertebrates and model organisms.

### Key Features
- **Genome Browser:** Interactive visualization of genomic regions
- **Gene annotation:** Comprehensive gene models with evidence
- **Comparative genomics:** Orthologs, paralogs, gene trees, whole-genome alignments
- **Variation data:** SNPs, structural variants, phenotype associations
- **Regulation:** Regulatory features, transcription factor binding sites

### BioMart
BioMart is Ensembl's bulk data retrieval tool. Use it when you need to:
- Download gene lists with annotations for a whole chromosome/genome
- Map between different identifier systems (Gene ID, transcript, protein)
- Get ortholog mappings between species

URL: https://www.ensembl.org/biomart/martview

In [ ]:
# Access Ensembl REST API
def ensembl_get(endpoint):
    """Query the Ensembl REST API."""
    url = f"https://rest.ensembl.org{endpoint}"
    req = urllib.request.Request(url, headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req) as response:
        return json.loads(response.read().decode())

# Look up a gene by symbol
gene = ensembl_get("/lookup/symbol/homo_sapiens/BRCA1?expand=1")

print(f"Gene: {gene['display_name']}")
print(f"Ensembl ID: {gene['id']}")
print(f"Biotype: {gene['biotype']}")
print(f"Location: {gene['seq_region_name']}:{gene['start']}-{gene['end']}")
print(f"Strand: {'+' if gene['strand'] == 1 else '-'}")
print(f"Description: {gene.get('description', 'N/A')}")

# List transcripts
transcripts = gene.get('Transcript', [])
print(f"\nTranscripts ({len(transcripts)}):")
for t in transcripts[:5]:
    print(f"  {t['id']} ({t['biotype']}) - {t['length']} bp")

In [ ]:
# Get the DNA sequence for a gene region from Ensembl
def ensembl_sequence(ensembl_id, seq_type="genomic"):
    """Get sequence for an Ensembl ID. seq_type: genomic, cds, cdna, protein."""
    url = f"https://rest.ensembl.org/sequence/id/{ensembl_id}?type={seq_type}"
    req = urllib.request.Request(url, headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req) as response:
        return json.loads(response.read().decode())

# Get the CDS sequence of a BRCA1 transcript
transcript_id = transcripts[0]['id']  # First transcript from previous query
cds = ensembl_sequence(transcript_id, seq_type="cds")

print(f"Transcript: {cds['id']}")
print(f"Sequence type: CDS")
print(f"Length: {len(cds['seq'])} bp")
print(f"First 120 nt: {cds['seq'][:120]}")

In [ ]:
# Find orthologs using Ensembl Compara
def get_orthologs(gene_id, target_species):
    """Get orthologs for a gene in a target species."""
    url = (f"https://rest.ensembl.org/homology/id/{gene_id}"
           f"?target_species={target_species}&type=orthologues")
    req = urllib.request.Request(url, headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req) as response:
        return json.loads(response.read().decode())

# Find mouse ortholog of human BRCA1
orthologs = get_orthologs("ENSG00000012048", "mus_musculus")

for homology in orthologs['data'][0]['homologies']:
    target = homology['target']
    print(f"Ortholog type: {homology['type']}")
    print(f"Mouse gene: {target['id']}")
    print(f"Species: {target['species']}")
    print(f"Protein identity: {target.get('perc_id', 'N/A')}%")

---
## 6. Summary: Choosing the Right Database

| I need... | Go to | Access method |
|-----------|-------|---------------|
| A nucleotide sequence | NCBI GenBank/RefSeq | Bio.Entrez |
| Curated protein info | UniProt/SwissProt | REST API |
| All protein sequences | UniProt/TrEMBL | REST API |
| 3D protein structure | PDB (RCSB) | REST API / download |
| Gene annotation in genomic context | Ensembl | REST API / BioMart |
| Raw sequencing reads | NCBI SRA | SRA Toolkit |
| Gene expression data | NCBI GEO | GEOquery / Entrez |
| Variant/mutation data | ClinVar, dbSNP | Entrez / VCF |
| Published papers | PubMed | Bio.Entrez |
| Ortholog relationships | Ensembl Compara | REST API |
| Bulk genome data | Ensembl BioMart | Web / biomaRt (R) |

---
## 7. Hands-on: Complete Workflow

Let us walk through a realistic scenario: given a gene name, find its sequence, protein structure, and key annotations.

In [ ]:
def investigate_gene(gene_name, organism="Homo sapiens"):
    """
    Given a gene name, retrieve key information from multiple databases.
    """
    print(f"Investigating {gene_name} ({organism})")
    print("=" * 60)
    
    # Step 1: Search NCBI for RefSeq mRNA
    print("\n[1] Searching NCBI nucleotide...")
    handle = Entrez.esearch(
        db="nucleotide",
        term=f"{gene_name}[Gene] AND {organism}[Organism] AND RefSeq[Filter] AND mRNA[Filter]",
        retmax=1
    )
    search_results = Entrez.read(handle)
    handle.close()
    
    if not search_results["IdList"]:
        print(f"  No RefSeq mRNA found for {gene_name}")
        return
    
    # Step 2: Fetch the GenBank record
    ncbi_id = search_results["IdList"][0]
    handle = Entrez.efetch(db="nucleotide", id=ncbi_id, rettype="gb", retmode="text")
    gb_record = SeqIO.read(handle, "genbank")
    handle.close()
    
    print(f"  Accession: {gb_record.id}")
    print(f"  Description: {gb_record.description}")
    print(f"  mRNA length: {len(gb_record)} bp")
    
    # Step 3: Extract CDS and translate
    print("\n[2] Extracting CDS...")
    for feat in gb_record.features:
        if feat.type == "CDS":
            cds_seq = feat.location.extract(gb_record.seq)
            protein = cds_seq.translate(to_stop=True)
            print(f"  CDS length: {len(cds_seq)} bp")
            print(f"  Protein length: {len(protein)} aa")
            print(f"  First 40 aa: {str(protein)[:40]}...")
            break
    
    # Step 4: Search UniProt
    print("\n[3] Searching UniProt...")
    try:
        org_id = "9606" if organism == "Homo sapiens" else ""
        up_results = search_uniprot(
            f"{gene_name} AND organism_id:{org_id} AND reviewed:true", limit=1
        )
        if up_results['results']:
            entry = up_results['results'][0]
            print(f"  UniProt ID: {entry['primaryAccession']}")
            prot_name = entry['proteinDescription']['recommendedName']['fullName']['value']
            print(f"  Protein name: {prot_name}")
    except Exception as e:
        print(f"  UniProt search failed: {e}")
    
    # Step 5: Search PDB
    print("\n[4] Searching PDB...")
    try:
        pdb_results = search_pdb(f"{gene_name} {organism}", max_results=3)
        n_structures = pdb_results.get('total_count', 0)
        print(f"  Found {n_structures} structures")
        for hit in pdb_results.get('result_set', [])[:3]:
            print(f"  PDB: {hit['identifier']}")
    except Exception as e:
        print(f"  PDB search failed: {e}")
    
    print("\n" + "=" * 60)
    print("Investigation complete.")

# Run the investigation
investigate_gene("TP53")

---
## 8. Exercises

### Exercise 1: Explore a Gene

Choose a gene of interest (suggestions: EGFR, BRCA2, SOD1, ACE2). Using `Bio.Entrez`:
1. Search the NCBI nucleotide database for its RefSeq mRNA
2. Download the GenBank record
3. Print: accession, organism, sequence length, number of features, and number of exons

In [ ]:
# Exercise 1: Your code here


### Exercise 2: Compare Databases

For the gene **SOD1** (superoxide dismutase 1):
1. Search NCBI protein database -- how many protein entries exist?
2. Search UniProt with `reviewed:true` -- how many SwissProt entries exist?
3. Search PDB -- how many crystal structures are available?

Why are the numbers so different across databases?

In [ ]:
# Exercise 2: Your code here


### Exercise 3: Protein Analysis from UniProt

Using the UniProt REST API:
1. Fetch the entry for human hemoglobin subunit beta (P68871)
2. Extract: protein name, length, subcellular location
3. List all PDB structures cross-referenced in this entry
4. Which PDB structure has the best resolution?

In [ ]:
# Exercise 3: Your code here


### Exercise 4: Batch Download

Write a function that:
1. Takes a list of GenBank accession numbers
2. Downloads all sequences in FASTA format (use a single `efetch` call with comma-separated IDs)
3. Saves them to a single FASTA file
4. Reports the total number of sequences and their lengths

Test with: `["NM_000207.3", "NM_001185098.2", "NM_000546.6"]` (insulin, insulin-like, TP53)

In [ ]:
# Exercise 4: Your code here


### Exercise 5: Cross-Database Links

Starting from the Gene ID **7157** (human TP53):
1. Use `Entrez.elink()` to find linked nucleotide records
2. Use `Entrez.elink()` to find linked protein records
3. Use `Entrez.elink()` to find linked PubMed articles
4. How many PubMed articles are linked to TP53?

Hint: `Entrez.elink(dbfrom="gene", db="pubmed", id="7157")`

In [ ]:
# Exercise 5: Your code here


---
## Cleanup

In [ ]:
import os
for f in ["4INS.pdb"]:
    if os.path.exists(f):
        os.remove(f)
        print(f"Removed {f}")
print("Cleanup complete.")

---
## Summary

**NCBI databases:**
- GenBank: submitted nucleotide sequences (uncurated)
- RefSeq: curated reference sequences (NM_, NP_, NC_ prefixes)
- Gene: gene-centric portal linking all NCBI data
- SRA: raw sequencing reads
- GEO: processed expression data

**UniProt:**
- SwissProt: manually reviewed proteins (~570K entries)
- TrEMBL: automatically annotated proteins (~250M entries)

**PDB:**
- Experimental 3D structures (X-ray, cryo-EM, NMR)
- Resolution: lower number = better detail

**Ensembl:**
- Genome annotation, comparative genomics, BioMart for bulk queries

**Programmatic access:**
- NCBI: `Bio.Entrez` (esearch, efetch, elink)
- UniProt: REST API at `rest.uniprot.org`
- PDB: REST API at `search.rcsb.org` and `data.rcsb.org`
- Ensembl: REST API at `rest.ensembl.org`

---

*Tier 2: Core Bioinformatics -- Biological Databases*

---

[← Previous: Bioinformatics Glossary: A-Z Reference](../../Tier_2_Core_Bioinformatics/00_Skills_Check/01_glossary.ipynb) | [Next: BioPython Essentials →](../../Tier_2_Core_Bioinformatics/02_BioPython_Essentials/01_biopython_essentials.ipynb)